In [41]:
import pandas as pd
import numpy as np

In [42]:
%run pull-current-records.ipynb
%run daily-sumo-picks.ipynb

C:\Users\willn\AppData\Local\Temp/ipykernel_10672/1030799868.py:17: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 17 of the file pull-current-records.ipynb. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  soup = BeautifulSoup(page)
C:\Users\willn\AppData\Local\Temp/ipykernel_10672/1030799868.py:17: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on l

In [43]:
## Points will be granted in the following ways:

## 1 pt. for each win
## 5 pts. for the championship or yusho
## 3 pts for the runner-up, or jun-yusho
## 3 pts for a special prize, or sansho
## 2 pts for a gold star victory, or kinboshi
## 1 pt. for a winning record, or kachi-koshi
## -.5 pt. for a losing record, or make-koshi
## -.5 pt. for your second trade
## -1 pt. for every trade beyond the first 2

In [44]:
month = '3'

In [45]:
year_short = '22'

In [46]:
picks = pd.read_csv(month+'-'+year_short+'-sumo-picks.csv')

In [47]:
cur_standings = pd.read_csv('current_makuuchi_standings.csv')

In [48]:
trades = pd.read_csv(month+'-'+year_short+'-trades.csv')

In [49]:
# Validate the current picks against the standings, 
#   sumodb standings as truth source
for pick in np.unique(picks.iloc[:,3:].values):
    if pick not in cur_standings.Rikishi.values:
        print(pick + ' is not in the standings')
        

In [50]:
for trade in np.unique(trades.iloc[:,2:].values):
    if trade not in cur_standings.Rikishi.values:
        print(trade + ' (trade) is not in the standings')

In [51]:
trades.shape

(26, 4)

In [52]:
def get_trades(row):
    row['trades'] = []
    if trades.shape[0] > 0:
        row['trades'] = trades[trades.Poster == row.Poster].values.tolist()
    return row

In [53]:
picks = picks.apply(get_trades,axis=1)

In [54]:
picks['num_trades'] = picks.trades.map(lambda x: len(x))

In [55]:
def get_points_hoshi(hoshitori,start=1,end=16):
    hoshitori = str(hoshitori)
    #print(hoshitori)
    kk = 0
    o_wins = hoshitori[start-1:end-1].count('O')
    #print(str(o_wins) + ' = o_wins')
    f_wins = hoshitori[start-1:end-1].count('%')
    #print(str(f_wins) + ' = f_wins')
    if o_wins+f_wins > 7:
        kk = 1
        #print('also kk')
    return o_wins + f_wins + kk

In [56]:
def check_mk(rikishi):
    mk = 0
    hoshitori = cur_standings[cur_standings.Rikishi == rikishi].Hoshitori.values[0]
    if len(hoshitori)-sum(map(hoshitori.count,['O','%'])) > 7:
        mk = -0.5
    return mk

In [57]:
def get_wins(rikishi,start=1,end=16):
    hoshitori = cur_standings[cur_standings.Rikishi == rikishi].Hoshitori.values[0]
    return get_points_hoshi(hoshitori,start,end)

In [58]:
def old_get_points(row):
    points = 0
    points += get_wins(row['YO'])
    points += get_wins(row['SK'])
    points += get_wins(row['M1-5'])
    points += get_wins(row['M6-10'])
    points += get_wins(row['M11+'])
    row['Points'] = points
    row['Points'] += row.trade_adj
    return row

In [59]:
def get_points_trade(day,traded,traded_for):
    points = 0
    points += get_wins(traded,start=1,end=day-1)
    points += get_wins(traded_for,start=day,end=15)
    return points

In [60]:
def get_points(row):
    if row.num_trades == 0:
        return old_get_points(row)
    row['Points'] = 0
    picks = row[['YO','SK','M1-5','M6-10','M11+']].tolist()
    traded = []
    for trade in row.trades:
        traded.append(trade[2])
        row['Points'] += get_points_trade(trade[1],trade[2],trade[3])
    for pick in picks:
        if pick in traded:
            continue
        row['Points'] += get_wins(pick)
    row['Points'] += row['trade_adj']
    return row

In [61]:
def trade_deduction(row):
    row['trade_adj'] = 0
    if row.num_trades < 2:
        return row
    if row.num_trades == 2:
        row.trade_adj = -0.5
        return row
    row.trade_adj = -((row.num_trades-2)+0.5)
    return row

In [62]:
def get_makekoshi(row):
    picks = row[['YO','SK','M1-5','M6-10','M11+']].tolist()
    for trade in row.trades:
        picks.append(trade[3])
    row['mk_adj'] = 0
    for pick in picks:
        row['mk_adj'] += check_mk(pick)
    row.Points += row.mk_adj
    return row

In [63]:
## Function to replace the names of the traded rikishi for the dashboard
def adjust_traded_view(row):
    for trade in row.trades:
        if row.YO == trade[2]:
            row.YO = trade[3]
        if row.SK == trade[2]:
            row.SK = trade[3]
        if row['M1-5'] == trade[2]:
            row['M1-5'] = trade[3]
        if row['M6-10'] == trade[2]:
            row['M6-10'] = trade[3]
        if row['M11+'] == trade[2]:
            row['M11+'] = trade[3]
    return row

In [64]:
picks = picks.apply(trade_deduction,axis=1)

In [65]:
picks = picks.apply(get_points,axis=1)

In [66]:
picks = picks.apply(get_makekoshi,axis=1)

In [67]:
picks = picks.apply(adjust_traded_view,axis=1)

In [68]:
picks[picks.gTier != 'Juryo'].sort_values('Points',ascending=False)

,Poster,gRank,gTier,YO,SK,M1-5,M6-10,M11+,trades,num_trades,trade_adj,Points,mk_adj
54,Teddybear,14.0,Maegashira,Mitakeumi,Wakatakakage,Endo,Kotonowaka,Kotoeko,"[[Teddybear, 4, Ishiura, Endo]]",1,0.0,52.5,-0.5
2,Beeswax,7.0,Maegashira,Mitakeumi,Abi,Tamawashi,Takayasu,Kotoshoho,[],0,0.0,50.5,-0.5
53,Takuan,1.0,Maegashira,Mitakeumi,Wakatakakage,Ura,Kotonowaka,Kotoshoho,[],0,0.0,50.5,-0.5
50,Skjorte,1.0,Sekiwake,Mitakeumi,Hoshoryu,Tamawashi,Kotonowaka,Kotoshoho,[],0,0.0,49.5,-0.5
35,Marching Powder,3.0,Maegashira,Mitakeumi,Abi,Endo,Takayasu,Kagayaki,[],0,0.0,49.5,-0.5
18,GoatSeeGuy,1.0,Ozeki,Mitakeumi,Hoshoryu,Ichinojo,Kotonowaka,Kotoshoho,"[[GoatSeeGuy, 6, Terunofuji, Mitakeumi]]",1,0.0,49.5,-0.5
25,Kenning,13.0,Maegashira,Takakeisho,Abi,Daieisho,Kotonowaka,Kotoshoho,[],0,0.0,49.0,0.0
56,Tiny Bug Child,1.0,Ozeki,Takakeisho,Abi,Daieisho,Kotonowaka,Kotoshoho,[],0,0.0,49.0,0.0
29,Lid,9.0,Maegashira,Mitakeumi,Abi,Ichinojo,Kotonowaka,Chiyomaru,[],0,0.0,47.5,-0.5
0,Banana Canada,2.0,Ozeki,Takakeisho,Abi,Daieisho,Takayasu,Kotokuzan,[],0,0.0,46.5,-0.5


In [69]:
pick_counts = picks[picks.gTier != 'Juryo']['M11+'].value_counts()

In [70]:
pick_counts = pick_counts.append(picks[picks.gTier != 'Juryo']['M6-10'].value_counts())

C:\Users\willn\AppData\Local\Temp/ipykernel_10672/1277004976.py:1: FutureWarning: The series.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  pick_counts = pick_counts.append(picks[picks.gTier != 'Juryo']['M6-10'].value_counts())


In [71]:
pick_counts = pick_counts.append(picks[picks.gTier != 'Juryo']['M1-5'].value_counts())

C:\Users\willn\AppData\Local\Temp/ipykernel_10672/4208511318.py:1: FutureWarning: The series.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  pick_counts = pick_counts.append(picks[picks.gTier != 'Juryo']['M1-5'].value_counts())


In [72]:
pick_counts = pick_counts.append(picks[picks.gTier != 'Juryo']['SK'].value_counts())

C:\Users\willn\AppData\Local\Temp/ipykernel_10672/965948409.py:1: FutureWarning: The series.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  pick_counts = pick_counts.append(picks[picks.gTier != 'Juryo']['SK'].value_counts())


In [73]:
pick_counts = pick_counts.append(picks[picks.gTier != 'Juryo']['YO'].value_counts())

C:\Users\willn\AppData\Local\Temp/ipykernel_10672/3186222200.py:1: FutureWarning: The series.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  pick_counts = pick_counts.append(picks[picks.gTier != 'Juryo']['YO'].value_counts())


In [74]:
# Y/O: Terunofuji
#S/K: Takanosho
#M1-5: Ura
#M6-10: Chiyonokuni
#M11+: Ichiyamamoto
picks[picks.Poster == 'Netsky'][['YO','SK','M1-5','M6-10','M11+']].values.tolist()[0]

['Mitakeumi', 'Wakatakakage', 'Ura', 'Kotonowaka', 'Myogiryu']

In [75]:
matchups = pd.read_csv('daily_matchups.csv')

In [76]:
def add_pick_counts(row):
    try:
        row['East_picks'] = pick_counts[row['East']]
    except:
        row['East_picks'] = 0
    try:
        row['West_picks'] = pick_counts[row['West']]
    except:
        row['West_picks'] = 0
    return row

matchups = matchups.apply(add_pick_counts,axis=1)

In [77]:
matchups = matchups.loc[:,['East_record','East','East_picks','West_picks','West','West_record']]

In [78]:
picks.Points.median()

44.5

In [79]:
picks[picks.Poster == 'Netsky'].Points

39    42.5
Name: Points, dtype: float64

In [80]:
poster = 'Netsky'
my_picks = picks[picks.Poster == poster][['YO','SK','M1-5','M6-10','M11+']].values.tolist()[0]

idx = pd.IndexSlice
slice_east_ = idx[matchups['East_picks'] < matchups['West_picks'], ['East']]
slice_west_ = idx[matchups['East_picks'] > matchups['West_picks'], ['West']]

def highlight_my_picks(s, props=''):
    return np.where(s.isin(my_picks), props, '')
matchups.style.set_properties(**{'background-color': 'lightgreen'}, subset=slice_east_).set_properties(**{'background-color': 'lightgreen'}, subset=slice_west_)\
                .apply(highlight_my_picks, props='color:white;background-color:green',subset=['East','West']).to_html('fantasy_sumo_dash.html',index=False)
